# Serengeti vocabulary with the masked-LM step, plus its control (kin, lug, swa, zul)

The first Serengeti-vocabulary run skipped the step the method prescribes: after new tokens get averaged vectors
(Fast Vocabulary Transfer), the embeddings are trained with a masked-LM pass on raw text before fine-tuning. This
notebook adds that step.

| model | how it is made | output (repo `results/`) |
|---|---|---|
| **Serengeti vocabulary + MLM** | `vocab_transfer.py`, then a masked-LM pass that trains only the input embeddings | `ner_serengeti_mlm`, `pos_serengeti_mlm` |
| **Control: AfroXLMR + MLM** | the same masked-LM pass (same text, steps, settings) on plain AfroXLMR | `ner_afroxlmr_mlm`, `pos_afroxlmr_mlm` |

The control matters: without it, a gain could come from extra masked-LM training on the four languages' text rather
than from the vocabulary. Both are then fine-tuned with the tuned LoRA from the sweep (same settings as the converged
baselines in `baselines_colab.ipynb`: up to 20 epochs, stop after 3 without dev gain), 3 seeds, NER and POS.

Raw text for the masked-LM pass: WURA (kin, swa, zul) and MADLAD-400 (lug), 20,000 blocks of 128 tokens per language,
3,000 steps of 32 blocks.

**Time on an L4 (estimate):** building both models 40 to 60 minutes, LoRA runs about 1 hour per model and task:
about 5 hours in total. Each (model, task) downloads a zip when it finishes.

**After a disconnect:** the adapted models live in `/content` and are rebuilt (the masked-LM pass is seeded but not
bit-identical on GPU). Remove finished entries from `RUNS` in step 2 so new runs do not mix with old ones.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > L4 GPU, then run again."

## 2. Settings and where results go

In [ ]:
RUNS = [("serengeti_mlm", "ner"), ("serengeti_mlm", "pos"), ("afroxlmr_mlm", "ner"), ("afroxlmr_mlm", "pos")]
USE_DRIVE = False
LANGS = "kin lug swa zul"
FVT_MODEL = "/content/afroxlmr_serengeti_fvt"
MODELS = {"serengeti_mlm": "/content/serengeti_fvt_mlm", "afroxlmr_mlm": "/content/afroxlmr_mlm"}

from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = Path("/content/drive/MyDrive/typology-aware-peft/results")
else:
    RESULTS = Path("/content/results")
RESULTS.mkdir(parents=True, exist_ok=True)
print("Results go to", RESULTS)

## 3. Get the code and the sweep's chosen LoRA setting (from GitHub)

In [ ]:
import os, shutil, json
REPO_URL = "https://github.com/WanjaWhoopie/typology-aware-peft.git"
WORK = Path("/content/typology-aware-peft")
if WORK.exists():
    !git -C {WORK} pull -q
else:
    !git clone -q {REPO_URL} {WORK}
os.chdir(WORK)
for name in ["src/token_baseline.py", "src/token_report.py", "src/vocab_transfer.py", "src/mlm_adapt.py",
             "src/morfessor_silver.py", "results/sweep/chosen.json"]:
    assert Path(name).exists(), f"{name} is not on GitHub yet (run the sweep first and push its results)"
CHOSEN = json.loads(Path("results/sweep/chosen.json").read_text())
LORA = {t: f"--lora_r {CHOSEN[t]['lora_r']} --lora_targets {CHOSEN[t]['lora_targets']}" for t in ["ner", "pos"]}
print("Chosen LoRA settings:", LORA)
!git log --oneline -1

## 4. Install packages

Same pinned versions as the SIB-200 run. Colab now ships an old `torchao` that makes peft crash when it
builds a LoRA model (this stopped every LoRA run on 2 Nov), so it is removed; nothing here uses it.

In [ ]:
!pip install -q transformers==4.57.2 datasets==4.4.1 peft==0.21.2 scikit-learn
!pip uninstall -y -q torchao
import transformers, datasets, peft
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| peft", peft.__version__)

## 5. Build the two models (40 to 60 minutes)

Prints the held-out masked-LM loss per language before and after the pass. It should fall clearly for the
Serengeti model (its new embeddings start as averages) and only a little for the control.

In [ ]:
needed = {m for m, _ in RUNS}
if "serengeti_mlm" in needed and not Path(MODELS["serengeti_mlm"], "model.safetensors").exists():
    if not Path(FVT_MODEL, "model.safetensors").exists():
        !python src/vocab_transfer.py --out {FVT_MODEL}
        assert _exit_code == 0, "vocabulary transfer failed"
    !python src/mlm_adapt.py --model {FVT_MODEL} --out {MODELS["serengeti_mlm"]}
    assert _exit_code == 0, "masked-LM pass failed"
if "afroxlmr_mlm" in needed and not Path(MODELS["afroxlmr_mlm"], "model.safetensors").exists():
    !python src/mlm_adapt.py --model Davlan/afro-xlmr-large-76L --out {MODELS["afroxlmr_mlm"]}
    assert _exit_code == 0, "masked-LM pass failed"
for m in needed:
    s = json.loads(Path(MODELS[m], "mlm_stats.json").read_text())
    print(m, {l: f"{s['heldout_loss_before'][l]:.2f} -> {s['heldout_loss_after'][l]:.2f}" for l in s["heldout_loss_after"]})

## 6. LoRA runs (about 1 hour per model and task)

**Out of memory:** add `--bs 8` and note it. Each (model, task) downloads e.g. `ner_serengeti_mlm_results.zip`;
the zip also holds the model's `mlm_stats.json`.

In [ ]:
from google.colab import files
for m, task in RUNS:
    name, out = f"{task}_{m}", RESULTS / f"{task}_{m}"
    print(f"===== {name} =====")
    !python src/token_baseline.py --task {task} --methods lora --langs {LANGS} --model {MODELS[m]} {LORA[task]} --epochs 20 --patience 3 --out {out}
    !python src/token_report.py --task {task} --dir {out} > /dev/null
    shutil.copy(Path(MODELS[m], "mlm_stats.json"), out / "mlm_stats.json")
    if not USE_DRIVE:
        shutil.make_archive(f"/content/{name}_results", "zip", out)
        files.download(f"/content/{name}_results.zip")

## 7. Done

Leave the zips in Downloads and tell Claude. They are compared with the converged LoRA baseline (`ner_conv`,
`pos_conv` from `baselines_colab.ipynb`), so run that notebook too.